In [ ]:
# app.py
from flask import Flask, request, jsonify
import face_recognition
import numpy as np
from PIL import Image
import io, os, pickle

app = Flask(__name__)
DATA_DIR = "face_db"
ENCODINGS_FILE = os.path.join(DATA_DIR, "encodings.pkl")

if not os.path.exists(DATA_DIR): os.makedirs(DATA_DIR)

# load known encodings
def load_encodings():
    if os.path.exists(ENCODINGS_FILE):
        with open(ENCODINGS_FILE, "rb") as f:
            return pickle.load(f)
    return {}  # name -> [encodings]

known = load_encodings()

def save_encodings():
    with open(ENCODINGS_FILE, "wb") as f:
        pickle.dump(known, f)

@app.route("/recognize", methods=["POST"])
def recognize():
    # accept raw image bytes (Content-Type image/jpeg)
    img_bytes = request.data
    if not img_bytes:
        return jsonify({"error":"no image"}), 400
    image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    arr = np.array(image)
    rgb = arr[:, :, ::-1]  # PIL gives RGB, face_recognition expects RGB actually - keep consistent

    face_locations = face_recognition.face_locations(rgb)
    face_encodings = face_recognition.face_encodings(rgb, face_locations)
    # default response
    for encoding in face_encodings:
        for name, enc_list in known.items():
            matches = face_recognition.compare_faces(enc_list, encoding, tolerance=0.5)
            if any(matches):
                return jsonify({"authorized": True, "id": name})
    # if none matched
    return jsonify({"authorized": False, "id": None})

@app.route("/enroll", methods=["POST"])
def enroll():
    # multipart form: file + name
    if 'file' not in request.files or 'name' not in request.form:
        return jsonify({"error":"missing file or name"}), 400
    f = request.files['file']
    name = request.form['name'].strip()
    img = Image.open(f).convert("RGB")
    arr = np.array(img)
    encoding = face_recognition.face_encodings(arr)
    if len(encoding) == 0:
        return jsonify({"error":"no face detected"}), 400
    enc = encoding[0]

    if name in known:
        known[name].append(enc)
    else:
        known[name] = [enc]
    save_encodings()
    return jsonify({"status":"ok", "id": name})

@app.route("/status", methods=["GET"])
def status():
    return jsonify({"uptime":"ok", "known": list(known.keys())})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)